# Classical SVM Classifier

Binary classification on the Iris dataset using a Support Vector Machine (SVM) with RBF kernel.

This notebook trains the classical baseline and saves results to `svm_results.json`  
for later use in `comparison/comparison.ipynb`.

**Configuration variable:** `N_FEATURES` controls how many features are used (2 or 4).  
For the 4-feature version, results are saved to `svm_results_4f.json` automatically.

## Imports

In [ ]:
import time
import json
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

## Configuration

Change `N_FEATURES` to 4 to run the 4-feature experiment.  
Everything downstream adapts automatically.

In [ ]:
N_FEATURES  = 2          # 2 or 4
TEST_SIZE   = 0.2        # fraction of data held out for testing
RANDOM_SEED = 42         # reproducibility

RESULTS_FILE = "svm_results.json" if N_FEATURES == 2 else "svm_results_4f.json"

print(f"Configuration: N_FEATURES={N_FEATURES}, TEST_SIZE={TEST_SIZE}")
print(f"Results will be saved to: {RESULTS_FILE}")

## Dataset

We use the Iris dataset restricted to **2 classes** (Versicolor vs Virginica, classes 1 and 2).  
Class 0 (Setosa) is dropped because it is linearly separable and trivially easy — it would
give an unrealistically high accuracy that obscures the real comparison.

Features used:
- `N_FEATURES = 2`: petal length, petal width (features 2 and 3 — most discriminative pair)
- `N_FEATURES = 4`: all four features (sepal length, sepal width, petal length, petal width)

All features are scaled to [0, 1] with `MinMaxScaler`. The quantum circuit will apply the
same scaling so that both models receive identical input.

In [ ]:
iris = load_iris()

# Keep only classes 1 and 2
mask     = iris.target != 0
X_full   = iris.data[mask]
y_full   = iris.target[mask] - 1      # remap to {0, 1}

# Feature selection
if N_FEATURES == 2:
    X = X_full[:, 2:4]                # petal length, petal width
    feature_names = ["petal length", "petal width"]
else:
    X = X_full                        # all 4 features
    feature_names = list(iris.feature_names)

# Scale to [0, 1]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_full, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_full
)

print(f"Total samples : {len(X_scaled)}")
print(f"Train samples : {len(X_train)}")
print(f"Test samples  : {len(X_test)}")
print(f"Features used : {feature_names}")
print(f"Class balance (train): {np.bincount(y_train)}")

## Classical Algorithm — SVM with RBF Kernel

A Support Vector Machine finds the hyperplane that maximizes the margin between classes.
The RBF (Radial Basis Function) kernel implicitly maps the input to a higher-dimensional
space, allowing non-linear boundaries.

This is the natural classical counterpart to the VQC: both are kernel-based methods,
the difference being that the VQC uses a quantum circuit to compute its kernel.

In [ ]:
def train_svm(X_train, y_train):
    """Train an SVM with RBF kernel. Returns the fitted model and elapsed training time."""
    model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_SEED)
    t0    = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - t0
    return model, elapsed


def evaluate_model(model, X_train, y_train, X_test, y_test):
    """Evaluate the model and return a metrics dict."""
    y_pred_test  = model.predict(X_test)
    y_pred_train = model.predict(X_train)

    acc_test  = accuracy_score(y_test,  y_pred_test)
    acc_train = accuracy_score(y_train, y_pred_train)

    # 5-fold cross-validation on the full scaled dataset
    cv_scores = cross_val_score(model, X_scaled, y_full, cv=5, scoring="accuracy")

    return {
        "accuracy_test"  : float(acc_test),
        "accuracy_train" : float(acc_train),
        "cv_mean"        : float(cv_scores.mean()),
        "cv_std"         : float(cv_scores.std()),
        "cv_scores"      : cv_scores.tolist(),
        "y_pred_test"    : y_pred_test.tolist(),
        "y_test"         : y_test.tolist(),
    }

## Run

In [ ]:
svm_model, train_time = train_svm(X_train, y_train)
metrics               = evaluate_model(svm_model, X_train, y_train, X_test, y_test)

print(f"Training time : {train_time:.4f} s")
print(f"Test accuracy : {metrics['accuracy_test']:.4f}")
print(f"Train accuracy: {metrics['accuracy_train']:.4f}")
print(f"CV accuracy   : {metrics['cv_mean']:.4f} ± {metrics['cv_std']:.4f}")
print()
print(classification_report(metrics['y_test'], metrics['y_pred_test'],
                             target_names=["Versicolor", "Virginica"]))

## Plots

In [ ]:
# --- Confusion matrix ---
cm = confusion_matrix(metrics['y_test'], metrics['y_pred_test'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Versicolor", "Virginica"])
disp.plot(colorbar=False)
plt.title(f"SVM — Confusion Matrix ({N_FEATURES} features)")
plt.tight_layout()
plt.savefig(f"svm_confusion_matrix{'_4f' if N_FEATURES == 4 else ''}.png", dpi=150)
plt.show()

In [ ]:
# --- Decision boundary (only for 2 features) ---
if N_FEATURES == 2:
    h = 0.005
    xx, yy = np.meshgrid(np.arange(0, 1 + h, h), np.arange(0, 1 + h, h))
    Z = svm_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_full,
                          cmap="coolwarm", edgecolors="k", s=40)
    plt.colorbar(scatter, ticks=[0, 1], label="Class")
    plt.xlabel(feature_names[0])
    plt.ylabel(feature_names[1])
    plt.title("SVM — Decision Boundary (2 features)")
    plt.tight_layout()
    plt.savefig("svm_decision_boundary.png", dpi=150)
    plt.show()
else:
    print("Decision boundary plot skipped (only available for N_FEATURES=2).")

## Save Results

All metrics are saved to `svm_results.json` (or `svm_results_4f.json` for 4 features).  
The comparison notebook loads this file directly — no need to re-run training.

In [ ]:
results = {
    # --- Experiment metadata ---
    "model"         : "SVM-RBF",
    "n_features"    : N_FEATURES,
    "feature_names" : feature_names,
    "n_train"       : int(len(X_train)),
    "n_test"        : int(len(X_test)),
    "random_seed"   : RANDOM_SEED,
    # --- Timing ---
    "train_time_s"  : float(train_time),
    # --- Accuracy ---
    "accuracy_test" : metrics["accuracy_test"],
    "accuracy_train": metrics["accuracy_train"],
    "cv_mean"       : metrics["cv_mean"],
    "cv_std"        : metrics["cv_std"],
    "cv_scores"     : metrics["cv_scores"],
    # --- Predictions (for confusion matrix in comparison notebook) ---
    "y_test"        : metrics["y_test"],
    "y_pred_test"   : metrics["y_pred_test"],
    # --- SVM-specific ---
    "n_support_vectors": int(sum(svm_model.n_support_)),
    "svm_C"            : svm_model.C,
    "svm_gamma"        : svm_model.gamma,
}

with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {RESULTS_FILE}")
print(json.dumps({k: v for k, v in results.items() if k not in ("cv_scores", "y_test", "y_pred_test")}, indent=2))